## Assignment 5

*100 points (7% of course grade)*</br>
*Assigned: Tue, Nov 17th*</br>
**Due: Mon, Dec 1st, 23:59**

This homework should be done in parts as soon as relevant topics are covered in lectures. If you wait until the last minute, you might be overwhelmed.

You must turn in the required files electronically, including this Notebook (A5.ipynb). Please follow the submission instructions for each problem carefully.

In this assignment, you need to solve three problems. In Problem 1, you will learn to create indexes to speed up query performance. In Problem 2, you will answer questions on some query processing algorithms. In Problem 3, you will answer questions on cardinality estimation.

### Setup your PostgreSQL

You will need this setup to create a database on your machine and to test your queries. Please follow our setup instructions with or without using the Docker Container on Canvas. You may reuse the Docker Container with Postgres from previous assignments.

### Problem 1: Indexing (40%)

In this task, you will be asked to i) select suitable indexes to speed up query performance and ii) examine the query plan of an SQL query. 

We are going to use a new database called `flights` attached in A5.zip. In the database, there is a single table, called flights. The following shows its schema:

**flights** (fid, year, month_id, day_of_month, day_of_week_id, carrier_id, flight_num, origin_city, origin_state, dest_city, dest_state, departure_delay, taxi_out, arrival_delay, canceled, actual_time, distance)

Note that in this task, you only need to use four attributes: `carrier_id`, `origin_city`, `actual_time`, and `dest_city`.

If you are going to run the commands in the Docker Container, you may want to copy the files using the command (remember to replace with your actual path):

```docker cp [A5] course_env:/root/A5```


Follow the steps to create and load the flights database: 
1. In your terminal, enter the `A5` folder (where the `flight_pg.sql` locates)
2. Create a database named flight (`createdb -U postgres flights`, or replace `postgres` with your username)
3. Run command: `psql -U postgres -d flights -f flight_pg.sql`,  or replace `postgres` with your username)
4. You may also import it with PgAdmin.
5. It may take a long time to insert all data. You may see errors because some columns have empty values, but it's fine.

Consider the following queries:

```sqlite
(Q1): SELECT DISTINCT carrier_id
     FROM Flights
     WHERE origin_city = 'Seattle WA' AND actual_time <= 180;
```


```sqlite
(Q2): SELECT DISTINCT carrier_id
     FROM Flights
     WHERE origin_city = 'Gunnison CO' AND actual_time <= 180;
```


```sqlite
(Q3): SELECT DISTINCT carrier_id
     FROM Flights
     WHERE origin_city = 'Seattle WA' AND actual_time <= 30;
```

##### (a). Choose one single simple index (index on **one attribute**) that is most likely to speed up all three queries. Write down the CREATE INDEX statement. (6 points)

In [ ]:
/* Input your answer in this cell */
CREATE INDEX index_city ON Flights(origin_city);

##### (b). Explain why you chose that index in (a). (include some statistics to support your choice) (6 points)


```Input your answer in this cell```

Because all 3 queries have conditions on origin_city. Equality conditions benefit the most from indexing compared to range conditions. Even though actual_time has more distinct values than origin_city (643 vs 327), it is used only in a range condition, which is less selective than an equality condition.


##### (c). Open a command line shell and start the PostgreSQL. Connect to the flights database, and add the index that you indicate above to the flights table. Please check whether each query used the index or not. Hint: you can use `EXPLAIN [QUERY]` (refer to the [documentation link](https://www.postgresql.org/docs/14/using-explain.html)) to see the query plan of each query. (6 points)

```Input your answer in this cell (only Yes/No)```

* **Did Query (Q1) use the index?**
    Yes

* **Did Query (Q2) use the index?**
    Yes
* **Did Query (Q3) use the index?**
    Yes


##### Consider this query:

```sql
(Q4): SELECT DISTINCT F2.origin_city
     FROM Flights F1, Flights F2
     WHERE F1.dest_city = F2.dest_city
         AND F1.origin_city='Gunnison CO'
         AND F1.actual_time <= 30;
```
##### (d). Choose one simple index (index on **one attribute**), different from the index for the question above, that is likely to speed up this query Q4. Write down the CREATE INDEX statement. (6 points)

In [ ]:
/* Input your answer in this cell */
CREATE INDEX index_dest ON flights(dest_city);

##### (e). Explain why you chose that index in (d) (include some statistics to support your choice).  (6 points)


``` Input your answer in this cell```

Because in Q4, there is a self-join on dest_city between F1 and F2. By adding an index on dest_city, it can efficiently look up all F2 tuples with the same destination city instead of scanning the entire Flights table, because the filters on F1 return a small number of rows so the main cost of this query comes from searching for matching rows in F2.


#### (f). Connect to the database flights, and check whether the flights table has this second index that you indicate above (Use `\d flights;` in your postgres). If not, add this index to the flights table. Then use the `EXPLAIN` command again to see the plan for Q4.

Did Query (Q4) use this second index? (2 points)

``` Input your answer in this cell (only Yes/No)```
Yes

##### Now we want to know how effective the two indexes are. We compare the runtime of the queries with and without indexes. Hint: use `\timing` in Postgres to turn SQL timer on ([Documentation](https://www.postgresql.org/docs/14/pgtesttiming.html)).

##### (g). Execute queries (Q1) - (Q4) on the flights table that **does not have** the two indexes (or drop the indexes). Please create a screenshot for the runtime of each query and put them in the `runtime` folder in the A5.zip, and name these figures as the img names in the cell below.  (4 points)

<table>
<tr>
    <td> <img src="runtime/no-indexes-a.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/no-indexes-b.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/no-indexes-c.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/no-indexes-d.png" alt="Drawing" style="width: 250px;"/> </td>
    </tr>
</table>

##### (h). Execute queries (Q1) - (Q4) on the flights table that **has the two indexes**. Please create a screenshot for the runtime of each query and put them in the `runtime` folder in the A5.zip, and name these figures as the img names in the cell below.  (4 points)

<table>
<tr>
    <td> <img src="runtime/with-indexes-a.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/with-indexes-b.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/with-indexes-c.png" alt="Drawing" style="width: 250px;"/> </td>
    <td> <img src="runtime/with-indexes-d.png" alt="Drawing" style="width: 250px;"/> </td>
    </tr>
</table>

### Problem 2. Query Processing Basics (36 points)
Recall the PalWorld database from previous assignments; now we consider a more complicated version. Human trainers capture Pals and can place them in production sites. Every Pal belongs to exactly one trainer (no "wild" Pals considered). We have the following tables:
* Trainer (tid, nickname): Each trainer has a unique id and a nickname.
* Pal (pid, species, suitability, level, tid): Each Pal has a unique id and belongs to one of the species. A Pal has exactly one work suitability (like Watering and Handiwork) with a level. Level is an integer in {1,2,3,4}. It has a trainer — tid is a foreign key referencing Trainer(tid).
* Site (sid, type, req_suitability): Each site has a unique sid and a type. Each site requires a work suitability for a Pal to work at this site, i.e., the Pal's suitability must be the same as the site’s req_suitability.
* WorkAt(pid, sid, start_time): Here, pid is a foreign key referencing Pal(pid) and sid is a foreign key referencing Site(sid). A record <pid, sid> means the Pal is currently working at Site from "start_time". A Pal cannot work at multiple Sites at the same time.

For this problem, we have following additional information:
* All rows in each table are stored as compactly as possible on disk blocks using the traditional n-ary storage model.
* $|Pal| = 10^5$, all Pal rows are sorted by pid, and each block holds $10$ Pal rows.
* $|Trainer| = 5*10^3$, all Trainer rows are sorted by by tid, and each block holds $20$ Trainer rows.


Answer the following questions. Note that one question may build on another; check each step carefully so you don’t get subsequent questions wrong because of a previous mistake.
* (a) How many blocks does Pal take? 
* (b) How many blocks does Trainer take? 

```input your answer for 2(a) and 2(b) in this cell:```

2(a) $10^5$ / 10 = $10^4$ = 10000 blocks

2(b) $5*10^3$ / 20 = 250 blocks


Suppose that we have a total of 21 memory blocks available to sort Pal by tid using the external merge sort algorithm described in lecture. (Any block used for buffering output must come from these blocks too.) For the following questions, if the algorithm already finished in Pass $k$, you should answer "N/A" for the number of level-$k'$ runs for any $k' > k$.
* (c) How many level-0 runs does the algorithm produce?
* (d) How many level-1 runs does the algorithm produce?
* (e) How many level-2 runs does the algorithm produce?
* (f) How many level-3 runs does the algorithm produce?
* (g) How many passes does the algorithm take? (Note that Pass $0$ counts as one pass too.)

```input your answer for 2(c) to 2(g) in this cell:```

2(c) ceiling (10000 / 21) ~ 476.19 = 477 level-0 runs

2(d) ceiling (477 / (21-1)) ~ 23.85 = 24 level-1 runs

2(e) ceiling (24 / 20) ~ 1.2 = 2 level-2 runs

2(f) ceiling (2 / 20) ~ 0.1 = 1 level-3 runs

2(g) 4 passes 


Suppose instead we have a total of $M=26$ memory blocks available to perform the two-pass hash join algorithm to compute $Pal \bowtie Trainer$. (Any block used for buffering output must come from these blocks too.) For simplicity, assume that all trainers have an equal number of Pal, and we have a perfect hash function.
* (h) What is the number of partitions per table the partitioning phase creates?
* (i) What is the size of a Pal partition (in blocks)?
* (j) What is the size of a Trainer partition (in blocks)?
* (k) In the probing phase, which table’s partitions (Pal or Trainer) must we use for the two-pass hash join to work?
* (l) Continuing with the above, suppose instead you get to pick $M$, the number of memory blocks available. What is the minimum value of $M$  required for the two-pass hash join algorithm to work? (using either formula from the lecture is fine)

```input your answer for 2(h) to 2(l) in this cell:```

(h) 26 - 1 = 25 partitions

(i) 10000 / 25 = 400 blocks

(j) 250 / 25 = 10 blocks

(k) pick the smaller table's partitions (Trainer). It fits into memory during the probing phase while Pal's partitions do not

(l) pick M > sqrt(min(B(Pal), B(trainer))) + 1

M > sqrt(min(10000,250)) + 1

M > 16.81

pick minimum M = 17 blocks


### Problem 3: Query Optimization Basics (24 points)

Consider again the four tables from the previous problem. Suppose that:
* $|Pal| = 10^5$, $|\pi_{species} Pal| = 10^2$
* $|Site| = 500$ and $|\pi_{type} Site| = 50$.
* $|WorkAt| = 4000 = |\pi_{pid} WorkAt|$ (suppose that only the most recent record of each Pal is stored) and $|\pi_{sid} WorkAt| = 500$.

Using the cardinality estimation techniques described in lecture, estimate the number of rows returned by the following queries.

* (a). $\sigma_{species = 'Splatterina'} Pal$

* (b). $(\sigma_{type = 'Sphere\ Assembly\ Line'} Site) \bowtie WorkAt$

* (c). $Pal \bowtie WorkAt$.

* (d). $\sigma_{species = 'Splatterina'} (Pal \bowtie WorkAt)$. Start with your answer for (c) and assume preservation of value set for $Pal.species$.

* (e). $(\sigma_{species = 'Splatterina'} Pal) \bowtie WorkAt$. Start with your answer for (a) and assume preservation of value set for $pid$.

* (f). You may notice that you got different answers for (d) and (e) above. But these two queries are equivalent! Discuss which one you feel to be more realistic (this is an open question).

```input your answer for 3(a) to 3(f) in this cell:```
* (a): $10^5$ / $10^2$ = $10^3$ = 1000
* (b): ( (500/50) * 4000 ) / max(50,500) = 40000 / 500 = 80
* (c): ($10^5$ * 4000) / max($10^5$,4000) = 4000 
* (d): 4000 / $10^2$ = 40 
* (e): 1000 * 4000 / max ($10^2$,4000) = 1000 
* (f): (e) seems more realistic because it starts with a smaller subset of Pals (those of species 'Splatterina') before joining with WorkAt, which is likely to yield a more accurate estimate than filtering after the join as in (d).

## Submission instruction

1. For all problem 1,2,3, answer the questions in the corresponding SQL/Markdown cells

2. Please do not add or remove cells.

3. Compress your A5.ipynb (this file) and your screenshots in `runtime` folder into A5.zip and submit on Canvas.